[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sapiosciences/sapio-py-tutorials/blob/master/04_custom_report.ipynb)

In [ ]:
from sapiopylib.rest.User import SapioUser
from sapiopylib.rest.DataMgmtService import DataMgmtServer
from sapiopylib.rest.pojo.CustomReport import *
from IPython.display import display, HTML
import pandas as pd
pd.set_option('display.max_columns', 100)

user = SapioUser(url="https://linux-vm:8443/webservice/api",
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7", account_name="sapio",
                 username="pyRestTest", password="Password1!", verify_ssl_cert=False)
customReportManager = DataMgmtServer.get_custom_report_manager(user)

# Custom Report (Advanced Search) API
The Custom Report API lets users send complex queries about data records across a hierarchy.

"Custom Report" is the API term for the "Advanced Search" feature that we present to users in the UI.

# Quick Report (Single Term)

In the first example, we run a quick report, which consists of a single term querying a data record.
We query for all samples with sample type 'Blood' in the system, limiting the output to the first 100 results.

We can obtain a pandas DataFrame from the report result in order to organize and display it.

*For paging to work, you must specify both the page size and the page number. If the page number is omitted, the report returns all results unpaged.*

In [ ]:
root_term = RawReportTerm('Sample', 'ExemplarSampleType', RawTermOperation.EQUAL_TO_OPERATOR, 'Blood')
report = customReportManager.run_quick_report(root_term, page_size=100, page_number=0)
data_frame = report.get_data_frame()
display(data_frame)

# Full Custom Report (Multiple Terms)
In a more advanced report, you may want to enter multiple conditions across a hierarchy of records.

Here is a way to get all attachments under the location '205 N George St'.

Note: You must pass a root term as the report term parameter. The terms under the root must form a binary tree. Any terms that are not connected to the root do not participate in the search.

Furthermore, for a data type to be part of a search, you must specify at least one term for it, even if that term is trivial, such as getting all records of type Attachment with RecordId >= 0.

In [ ]:
term1 = RawReportTerm('VeloxLocation', 'LocationName', RawTermOperation.EQUAL_TO_OPERATOR, '205 N George St')
term2 = RawReportTerm('Attachment', 'RecordId', RawTermOperation.GREATER_THAN_OR_EQUAL_OPERATOR, '0')
root_term = CompositeReportTerm(term1, CompositeTermOperation.AND_OPERATOR, term2)

column_list = [ReportColumn('Attachment', 'FilePath', FieldType.STRING), ReportColumn('Attachment', 'RecordId', FieldType.LONG)]

request = CustomReportCriteria(column_list, root_term, page_size=1, page_number=0)

report = customReportManager.run_custom_report(request)
data_frame = report.get_data_frame()
display(data_frame)

# Inner Join, Hierarchically Unrelated Records
Since Sapio 22.10, we can add **explicit joins** between records of types that are **not hierarchically related**.

In this example, let's find the sample's receipt status along with the occupancy information for the storage unit containing sample "00001".

Observe that in *ExplicitJoinDefinition*, the data type name outside of the join term must be set to the **Unrelated Data Type Name** in the report. In this case, storage units are not related to either samples or sample receipts via a hierarchy.

In [ ]:
sample_term = RawReportTerm('Sample', 'SampleId', RawTermOperation.EQUAL_TO_OPERATOR, '00001')
receipt_term = RawReportTerm('SampleReceipt', 'RecordId', RawTermOperation.GREATER_THAN_OPERATOR, '0')
root_term = CompositeReportTerm(sample_term, CompositeTermOperation.AND_OPERATOR, receipt_term)


column_list = [ReportColumn('Sample', 'SampleId', FieldType.STRING), ReportColumn('SampleReceipt', 'SampleReceivedRejected', FieldType.PICKLIST),
               ReportColumn('StorageUnit', 'OccupiedCount', FieldType.LONG), ReportColumn('StorageUnit', 'StorageUnitCapacity', FieldType.LONG)]

storage_join_term = FieldCompareReportTerm('Sample', 'StorageLocationBarcode', RawTermOperation.EQUAL_TO_OPERATOR, 'StorageUnit', 'StorageUnitId')

request = CustomReportCriteria(column_list=column_list, root_term=root_term, join_list=[ExplicitJoinDefinition('StorageUnit', storage_join_term)])

report = customReportManager.run_custom_report(request)
data_frame = report.get_data_frame()
display(data_frame)